In [ ]:
import importlib.util
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, HTML
import subprocess
import sys
import time

# Package Verification & Installation Setup
packages_to_check = [
    "optuna",
    "tqdm",
    "gdown",
    "catboost",
    "scikit-learn",
]

missing_packages = [
    pkg for pkg in packages_to_check
    if importlib.util.find_spec(pkg) is None
]

progress = widgets.IntProgress(
    value=0,
    min=0,
    max=100,
    description="Setup:",
    bar_style="info"
)

status = widgets.HTML(value="Starting setup...")
display(progress, status)

progress.value = 20
status.value = "Checking required packages..."

if missing_packages:
    step_val = 70 / len(missing_packages)

    for i, pkg in enumerate(missing_packages):
        status.value = f"Installing package: {pkg}..."

        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", pkg],
            check=True
        )

        progress.value = int(20 + (i + 1) * step_val)

progress.value = 100
progress.bar_style = "success"
status.value = "✓ All required packages are available."

# Imports
import numpy as np
import optuna
import pandas as pd

from tqdm.notebook import tqdm

from catboost import CatBoostClassifier

from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    cross_val_score,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
# 1. Load data
DATA_FILE = Path("DATA") / "data_clean.csv"
df = pd.read_csv(DATA_FILE)

X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# 2. Column definitions
question_cols = [
    'N1', 'N2', 'N3', 'N4', 'N5', 'N6', 'N7', 'N8', 'N9', 'N10',
    'E1', 'E3', 'E4', 'E5', 'E7', 'E9', 'E10', 'C4', 'A4'
]
numeric_cols = question_cols + ['age']
categorical_cols = ['gender', 'hand']

# 2.5 Preprocessing
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
]) 

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
]) 

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

In [ ]:
# 3.1. Pipeline for Logistic Regression
pipe_logreg = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)

# 3.2. Pipeline for KNN
pipe_knn = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", KNeighborsClassifier()),
    ]
)

# 3.3. Pipeline for Random Forest
pipe_rf = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(random_state=42)),
    ]
)

# 3.4. Pipeline for Gradient Boosting
pipe_hgb = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", HistGradientBoostingClassifier(random_state=42)),
    ]
)
# 3.4. Pipeline for Gradient Boosting
pipe_hgb = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", HistGradientBoostingClassifier(random_state=42)),
    ]
)

# 3.4b. Pipeline for Gradient Boosting (Tuned variant)
pipe_hgb_tuned = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", HistGradientBoostingClassifier(random_state=42)),
    ]
)

# 3.5. Pipeline for CatBoost
pipe_cat = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", CatBoostClassifier(random_state=42, verbose=0)),
    ]
)

## Base Model Evaluation (Without Tuning)

In [ ]:
base_pipelines = {
    "Logistic Regression": pipe_logreg,
    "K-Nearest Neighbors": pipe_knn,
    "Random Forest": pipe_rf,
    "Gradient Boosting": pipe_hgb,
    "CatBoost": pipe_cat
}
base_results = []
base_results_html = ""

for name, pipe in tqdm(base_pipelines.items(), desc="Training Base Models"):
    start_time = time.time()
    pipe.fit(X_train, y_train)
    elapsed_time = time.time() - start_time
    
    fold_scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='f1_macro', n_jobs=-1)
    mean_cv = fold_scores.mean()
    std_cv = fold_scores.std()
    test_score = pipe.score(X_test, y_test)
    
    base_results.append({
        "name": name,
        "model": pipe,
        "test_score": test_score,
        "cv_score": mean_cv
    })
    
    folds_html = "".join(
        f"""
        <div style="background:#f8fafc; border:1px solid #e2e8f0; border-radius:6px; padding:8px; text-align:center;">
            <div style="color:#64748b; font-size:11px;">Fold {i}</div>
            <div style="font-size:16px; font-weight:700; color:#2563eb;">{score:.3f}</div>
        </div>
        """
        for i, score in enumerate(fold_scores, start=1)
    )

    base_results_html += f"""
    <div style="background:#f8fafc; border:1px solid #e2e8f0; border-radius:10px; padding:16px; margin-bottom:14px;">
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
            <h4 style="margin:0; color:#1e293b; font-size:18px;">{name} (Default)</h4>
            <span style="color:#64748b; font-size:13px;">Duration: <b>{elapsed_time:.2f} sec</b></span>
        </div>
        <div style="font-size:12px; color:#475569; margin-bottom:10px;">
            <b>Parameters:</b> Default scikit-learn settings
        </div>
        <div style="display:flex; gap:12px; margin-bottom:12px;">
            <div style="background:#eff6ff; border-radius:8px; padding:10px; flex:1;">
                <div style="color:#475569; font-size:11px;">CV-Score (Ø F1 Macro)</div>
                <div style="font-size:20px; font-weight:700; color:#1d4ed8;">{mean_cv:.3f} <span style="font-size:12px; color:#64748b; font-weight:normal;">(±{std_cv:.3f})</span></div>
            </div>
            <div style="background:#f0fdf4; border-radius:8px; padding:10px; flex:1;">
                <div style="color:#475569; font-size:11px;">Test-Score (F1 Macro)</div>
                <div style="font-size:20px; font-weight:700; color:#15803d;">{test_score:.3f}</div>
            </div>
        </div>
        <div style="display:grid; grid-template-columns:repeat(5, 1fr); gap:6px;">
            {folds_html}
        </div>
    </div>
    """

best_base_overall = max(base_results, key=lambda x: x["test_score"])

display(HTML(f"""
<div style="font-family:Arial, sans-serif; max-width:850px; margin:16px 0;">
    <h3 style="margin-bottom:12px;">Base Model Comparison (Without Tuning) & F1 Macro Details</h3>
    {base_results_html}
    
    <div style="background:#fefce8; border:1px solid #fde047; border-radius:10px; padding:16px; margin-top:16px;">
        <h4 style="margin:0 0 6px 0; color:#854d0e; font-size:18px;">🏆 Best Base Model: {best_base_overall['name']}</h4>
        <div style="font-size:13px; color:#713f12;">
            The model achieved the highest performance with a <b>Test-Score (F1 Macro) of {best_base_overall['test_score']:.3f}</b>.
        </div>
    </div>
</div>
"""))

In [ ]:
# 4. Dictionary for GridSearch Tuning & Models Configuration
models_config = {
    "Logistic Regression": {
        "pipeline": pipe_logreg,
        "params": {
            "classifier__C": [0.01, 0.1, 1, 10],
            "classifier__solver": ["lbfgs"],
        },
    },
    "KNN": {
        "pipeline": pipe_knn,
        "params": {
            "classifier__n_neighbors": [3, 5, 7, 9],
            "classifier__weights": ["uniform", "distance"],
        },
    },
    "Random Forest": {
        "pipeline": pipe_rf,
        "params": {
            "classifier__n_estimators": [100, 200],
            "classifier__max_depth": [None, 10, 20],
            "classifier__min_samples_split": [2, 5],
        },
    },
    "HistGradient Boosting": {
        "pipeline": pipe_hgb,
        "params": {
            "classifier__learning_rate": [0.01, 0.05, 0.1],
            "classifier__max_iter": [100, 200],
        },
    },
    "HistGradient Boosting Tuned": {
        "pipeline": pipe_hgb_tuned,
        "params": {
            "classifier__max_leaf_nodes": [31, 32],
            "classifier__min_samples_leaf": [20, 60],
            "classifier__class_weight": ["balanced"],
        },
    },

    # "CatBoost": {
    #     "pipeline": pipe_cat,
    #     "params": {
    #         "classifier__iterations": [400, 500, 600],
    #         "classifier__depth": [5, 6, 7],
    #         "classifier__learning_rate": [0.03, 0.05, 0.07],
    #         "classifier__l2_leaf_reg": [4, 6, 8]
    #     }
    # }
}

## Hyperparameter Tuning & Model Comparison

In [ ]:
# 5. Automated execution with progress bar, tuning, folds & HTML generation
results = []
results_html = ""

for name, config in tqdm(models_config.items(), desc="Training Models"):
    start_time = time.time()
    
    # GridSearchCV for hyperparameter tuning using f1_macro
    grid_search = GridSearchCV(
        estimator=config["pipeline"],
        param_grid=config["params"],
        cv=5,
        scoring='f1_macro',
        n_jobs=-1
    )
    grid_search.fit(X_train, y_train)
    elapsed_time = time.time() - start_time
    
    best_model = grid_search.best_estimator_
    
    # Calculate fold scores for the best model visualization
    fold_scores = cross_val_score(best_model, X_train, y_train, cv=5, scoring='f1_macro', n_jobs=-1)
    mean_cv = fold_scores.mean()
    std_cv = fold_scores.std()
    
    test_score = best_model.score(X_test, y_test)
    
    results.append({
        "name": name,
        "model": best_model,
        "test_score": test_score,
        "cv_score": mean_cv
    })
    
    # Generate HTML for the 5 folds
    folds_html = "".join(
        f"""
        <div style="background:#f8fafc; border:1px solid #e2e8f0; border-radius:6px; padding:8px; text-align:center;">
            <div style="color:#64748b; font-size:11px;">Fold {i}</div>
            <div style="font-size:16px; font-weight:700; color:#2563eb;">{score:.3f}</div>
        </div>
        """
        for i, score in enumerate(fold_scores, start=1)
    )
    
    best_params_str = str(grid_search.best_params_).replace("{", "").replace("}", "").replace("'", "")

    # Build HTML card component
    results_html += f"""
    <div style="background:#f8fafc; border:1px solid #e2e8f0; border-radius:10px; padding:16px; margin-bottom:14px;">
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
            <h4 style="margin:0; color:#1e293b; font-size:18px;">{name}</h4>
            <span style="color:#64748b; font-size:13px;">Duration: <b>{elapsed_time:.2f} sec</b></span>
        </div>
        <div style="font-size:12px; color:#475569; margin-bottom:10px;">
            <b>Best Parameters:</b> {best_params_str}
        </div>
        <div style="display:flex; gap:12px; margin-bottom:12px;">
            <div style="background:#eff6ff; border-radius:8px; padding:10px; flex:1;">
                <div style="color:#475569; font-size:11px;">CV-Score (Ø F1 Macro)</div>
                <div style="font-size:20px; font-weight:700; color:#1d4ed8;">{mean_cv:.3f} <span style="font-size:12px; color:#64748b; font-weight:normal;">(±{std_cv:.3f})</span></div>
            </div>
            <div style="background:#f0fdf4; border-radius:8px; padding:10px; flex:1;">
                <div style="color:#475569; font-size:11px;">Test-Score (F1 Macro)</div>
                <div style="font-size:20px; font-weight:700; color:#15803d;">{test_score:.3f}</div>
            </div>
        </div>
        <div style="display:grid; grid-template-columns:repeat(5, 1fr); gap:6px;">
            {folds_html}
        </div>
    </div>
    """

# Determine the best overall model based on test score
best_overall = max(results, key=lambda x: x["test_score"])
best_pipeline = best_overall["model"]

display(HTML(f"""
<div style="font-family:Arial, sans-serif; max-width:850px; margin:16px 0;">
    <h3 style="margin-bottom:12px;">Model Comparison, Tuning & F1 Macro Details</h3>
    {results_html}
    
    <div style="background:#fefce8; border:1px solid #fde047; border-radius:10px; padding:16px; margin-top:16px;">
        <h4 style="margin:0 0 6px 0; color:#854d0e; font-size:18px;">🏆 Best Model: {best_overall['name']}</h4>
        <div style="font-size:13px; color:#713f12;">
            The model achieved the highest performance with a <b>Test-Score (F1 Macro) of {best_overall['test_score']:.3f}</b>.
        </div>
    </div>
</div>
"""))

## Head-to-Head: Base Winner vs. Tuned Winner

In [ ]:
# Best overall across both categories
absolute_best = max([best_base_overall, best_overall], key=lambda x: x["test_score"])

# Styling variables: Winner gets thick green border & green bg, loser gets subtle blue tint & standard border
base_is_winner = best_base_overall["test_score"] >= best_overall["test_score"]

base_border = "3px solid #16a34a" if base_is_winner else "1px solid #cbd5e1"
base_bg = "#dcfce7" if base_is_winner else "#f8fafc"

tuned_border = "3px solid #16a34a" if not base_is_winner else "1px solid #cbd5e1"
tuned_bg = "#dcfce7" if not base_is_winner else "#eff6ff"

display(HTML(f"""
<div style="font-family:Arial, sans-serif; max-width:850px; margin:16px 0;">
    <div style="display:flex; gap:16px; margin-bottom:16px;">
        <!-- Base Winner Card -->
        <div style="background:{base_bg}; border:{base_border}; border-radius:10px; padding:16px; flex:1;">
            <div style="color:#64748b; font-size:12px; font-weight:bold; text-transform:uppercase; margin-bottom:4px;">Base Winner</div>
            <h4 style="margin:0 0 10px 0; color:#1e293b; font-size:18px;">{best_base_overall['name']}</h4>
            <div style="font-size:13px; color:#475569; margin-bottom:8px;">
                <b>Test-Score (F1):</b> <span style="color:#15803d; font-weight:700;">{best_base_overall['test_score']:.3f}</span>
            </div>
            <div style="font-size:13px; color:#475569;">
                <b>CV-Score (F1):</b> {best_base_overall['cv_score']:.3f}
            </div>
        </div>

        <!-- Tuned Winner Card -->
        <div style="background:{tuned_bg}; border:{tuned_border}; border-radius:10px; padding:16px; flex:1;">
            <div style="color:#64748b; font-size:12px; font-weight:bold; text-transform:uppercase; margin-bottom:4px;">Tuned Winner</div>
            <h4 style="margin:0 0 10px 0; color:#1e293b; font-size:18px;">{best_overall['name']}</h4>
            <div style="font-size:13px; color:#475569; margin-bottom:8px;">
                <b>Test-Score (F1):</b> <span style="color:#15803d; font-weight:700;">{best_overall['test_score']:.3f}</span>
            </div>
            <div style="font-size:13px; color:#475569;">
                <b>CV-Score (F1):</b> {best_overall['cv_score']:.3f}
            </div>
        </div>
    </div>
    
    <div style="background:#fefce8; border:1px solid #fde047; border-radius:10px; padding:16px; margin-bottom:16px;">
        <h4 style="margin:0 0 6px 0; color:#854d0e; font-size:18px;">🏆 Absolute Winner: {absolute_best['name']}</h4>
        <div style="font-size:13px; color:#713f12;">
            The model achieved the best overall performance across all evaluated variants with a <b>Test-Score of {absolute_best['test_score']:.3f}</b>.
        </div>
    </div>

    <div style="background:#f8fafc; border:1px solid #e2e8f0; border-radius:10px; padding:16px;">
        <h4 style="margin:0 0 6px 0; color:#1e293b; font-size:16px;">💡 Comparison & Trade-off</h4>
        <div style="font-size:13px; color:#475569; line-height:1.5;">
            While base models train quickly using default parameters, hyperparameter tuning systematically explores regularizations and structural constraints. This comparison highlights which variant ultimately proved optimal on the test set.
        </div>
    </div>
</div>
"""))

## Model Persistence (Joblib)

In [ ]:
import joblib

# Define filename for the best model
model_filename = "best_model.joblib"

# Save the best pipeline / model to disk
joblib.dump(best_pipeline, model_filename)

# Display a stylish confirmation card in the notebook
display(HTML(f"""
<div style="font-family:Arial, sans-serif; max-width:850px; margin:16px 0; background:#f0fdf4; border:1px solid #bbf7d0; border-radius:10px; padding:16px;">
    <h4 style="margin:0 0 6px 0; color:#166534; font-size:18px;">💾 Model Successfully Saved</h4>
    <div style="font-size:13px; color:#14532d; line-height:1.5;">
        The winning pipeline for <b>{best_overall['name']}</b> has been serialized and saved to 
        <code>{model_filename}</code> using <b>Joblib</b>. It is ready to be loaded for inference or deployment.
    </div>
</div>
"""))